In [9]:
import os
import sys

# Ensure the workspace root is on Python's import path for notebook execution.
sys.path.insert(0, os.path.abspath(os.getcwd()))

from dotenv import load_dotenv
load_dotenv(".env")

True

In [11]:
%load_ext autoreload
%autoreload 2

from email_assistant.eval.email_dataset import email_inputs,expected_tool_calls,triage_outputs_list, response_criteria_list, examples_triage
test_case_ix = 0

print("Email Input:",email_inputs[test_case_ix])
print("Expected traige output:",triage_outputs_list[test_case_ix])
print("Expected tool calls:",expected_tool_calls[test_case_ix])
print("Response Criteria:", response_criteria_list[test_case_ix])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Email Input: Subject: Invoice payment

Hi team, I have a question about my latest invoice.
Expected traige output: This looks like a billing inquiry that should be routed to Finance.
Expected tool calls: ['open_invoice', 'check_payment_status']
Response Criteria: Provide a clear statement of the issue and next steps for the user.


In [12]:
import pytest
from email_assistant.eval.email_dataset import email_inputs,expected_tool_calls
from email_assistant.utils import format_messages_string, extract_tool_calls, extract_toll_calls
from email_assistant.email_assistant import email_assistant

from langsmith import testing as t

@pytest.mark.langsmith
@pytest.mark.parametrize(
    "email_input,expected_calls",
    [
         (email_inputs[0],expected_tool_calls[0]),
         (email_inputs[3],expected_tool_calls[3]),
    ],
)

def test_email_dataset_tool_call(email_input,expected_calls):
    """Test if email processing contains expected tool calls.
    
    This test confirms that all expected tools are called during email processing,
    but does not check the order of tool invocations or the number of invocations
    per tool. Additional checks for these aspects could be added if desired.
    """

    # run the email assiaant
    messages = [{"role": "user", "content": str(email_input)}]
    result = email_assistant.invoke({"messages": messages})

    extracted_tool_calls = extract_tool_calls(result['messages'])

    missing_calls = [call for call in expected_calls if call.lower() not in extracted_tool_calls]
    t.log_outputs({
                "missing_calls": missing_calls,
                "extracted_tool_calls": extracted_tool_calls,
                "response": format_messages_string(result['messages'])
            })

    # Test passes if no expected calls are missing
    assert len(missing_calls) == 0






In [13]:
import os
from dotenv import load_dotenv

# Try loading from parent directory
loaded = load_dotenv("../.env")
print("Loaded .env:", loaded)

# Verify the key is present
api_key = os.getenv("LANGSMITH_API_KEY")
if not api_key:
    raise EnvironmentError("❌ LANGSMITH_API_KEY not found. Check your .env file path and contents.")
else:
    print(f"✅ LANGSMITH_API_KEY found: {api_key[:8]}...")

Loaded .env: False
✅ LANGSMITH_API_KEY found: lsv2_pt_...


In [14]:
import os
from langsmith import Client

from email_assistant.eval.email_dataset import examples_triage

# Explicitly pass the API key to avoid relying on env vars being set
api_key = os.getenv("LANGSMITH_API_KEY")
if not api_key:
    raise EnvironmentError("❌ LANGSMITH_API_KEY is not set. Please check your .env file.")

# Initialize LangSmith client with explicit key
client = Client(api_key=api_key)

# Dataset name
dataset_name = "E-mail Triage Evaluation"

# Create dataset if it doesn't exist
try:
    if not client.has_dataset(dataset_name=dataset_name):
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="A dataset of e-mails and their triage decisions."
        )
        # Add examples to the dataset
        client.create_examples(dataset_id=dataset.id, examples=examples_triage)
        print(f"✅ Dataset '{dataset_name}' created with {len(examples_triage)} examples.")
    else:
        print(f"✅ Dataset '{dataset_name}' already exists.")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

✅ Dataset 'E-mail Triage Evaluation' created with 4 examples.


In [29]:
from email_assistant.eval.email_dataset import examples_triage

print("Dataset Example Item:", examples_triage[0])
print("Dataset Example Input (inputs):", examples_triage[0].get('inputs'))
print("Dataset Example Reference output (outputs):", examples_triage[0].get('outputs'))

Dataset Example Item: {'inputs': {'email_input': 'Subject: Invoice payment\n\nHi team, I have a question about my latest invoice.'}, 'outputs': {'classification': 'respond', 'expected_tool_calls': ['open_invoice', 'check_payment_status'], 'triage_output': 'This looks like a billing inquiry that should be routed to Finance.', 'response_criteria': 'Provide a clear statement of the issue and next steps for the user.'}}
Dataset Example Input (inputs): {'email_input': 'Subject: Invoice payment\n\nHi team, I have a question about my latest invoice.'}
Dataset Example Reference output (outputs): {'classification': 'respond', 'expected_tool_calls': ['open_invoice', 'check_payment_status'], 'triage_output': 'This looks like a billing inquiry that should be routed to Finance.', 'response_criteria': 'Provide a clear statement of the issue and next steps for the user.'}


In [30]:
print("Dataset Example Reference Output (reference output): ",examples_triage[0]['output'])

KeyError: 'output'

In [31]:
def target_email_assistant(inputs:dict) -> dict:
    """Process an email through the workflow-based email assistant."""
    response = email_assistant.nodes['triage_router'].invoke({"email_input": inputs["email_input"]})
    return {"classification_decision": response.update['classification_decision']}

In [32]:
def classification_evaluator(outputs:dict,refrence_outputs:dict) -> bool:
    """Check if the answer exactly matches the expected answer."""
    return outputs["classification_decision"].lower() == refrence_outputs["classification"].lower()

In [33]:
run_expt = True
if run_expt:
    experiment_results_workflow = client.evaluate(
        target_email_assistant,
        data = dataset_name,
        evaluators = [classification_evaluator],
        experiment_prefix = "E-mail assistant workflow",
        max_concurrency = 2,
    )

StopIteration: 

In [ ]:
from pydantic import BaseModel,Field
from langchain.chat_models import init_chat_model
class CriteriaGrade(BaseModel):
    """Score the response against specific criteria."""
    justification: str = Field(description="The justification for the grade and score, including specific examples from the response.")
    grade: bool = Field(description="Does the response meet the provided criteria?")

criteria_eval_llm = init_chat_model("gemma4")
criteria_eval_structured_llm = criteria_eval_llm.with_structured_output(CriteriaGrade)    

In [ ]:
email_input = email_inputs[0]
print("Email Input:",email_input)
success_criteria = response_criteria_list[0]
print("success_criteria:",success_criteria)

In [ ]:
response = email_assistant.invoke({"email_input":email_input})



In [ ]:
from email_assistant.eval.prompts import RESPONSE_CRITERIA_SYSTEM_PROMPT

all_messages_str = format_messages_string(response['messages'])
eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {success_criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])

eval_result

In [ ]:
RESPONSE_CRITERIA_SYSTEM_PROMPT

In [ ]:
# TODO: Copy your experiment name here
experiment_name = "email_assistant:8286b3b8"
# Set this to load expt results
load_expt = False
if load_expt:
    email_assistant_experiment_results = client.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", email_assistant_experiment_results.latency_p50)
    print("Latency p99:", email_assistant_experiment_results.latency_p99)
    print("Token Usage:", email_assistant_experiment_results.total_tokens)
    print("Feedback Stats:", email_assistant_experiment_results.feedback_stats)